# CUHK03
We want to save the dataset in a standard format, to be used with our models. 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
# Download CUHK03 dataset from Kaggle
!curl -L -o /content/drive/MyDrive/AML-Project/cuhk03.zip\
https://www.kaggle.com/api/v1/datasets/download/priyanagda/cuhk03


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2757M  100 2757M    0     0  48.7M      0  0:00:56  0:00:56 --:--:-- 56.1M


In [2]:
# Unzip the file
!unzip /content/drive/MyDrive/AML-Project/cuhk03.zip -d /content/drive/MyDrive/AML-Project/cuhk03

Streaming output truncated to the last 5000 lines.
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_094_2_10.png  
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_095_1_01.png  
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_095_1_02.png  
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_095_1_03.png  
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_095_1_04.png  
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_095_1_05.png  
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_095_2_06.png  
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_095_2_07.png  
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_095_2_08.png  
  inflating: /content/drive/MyDrive/AML-Project/cuhk03/archive/images_labeled/2_095_2_09.png  

In [2]:
import os
import json
import shutil
import random

# Set seed for reproducibility
random.seed(2)

ds_path = "./cuhk03/archive"
converted_path = "./cuhk03/converted-split2"
split_json = os.path.join(ds_path, "splits_classic_labeled.json")

In [42]:
with open(split_json, "r") as file:  
    data = json.load(file)

query = data[0].get('query')
gallery = data[0].get('query')

print(f"Query has {len(query)} elements")
print(f"Gallery has {len(gallery)} elements")

Query has 965 elements
Gallery has 965 elements


In [43]:
# CUHK ID -> Custom ID
ids_map = {}
new_id = 0

def convert(entries, f_type):
    assert f_type in ["gallery", "query"]

    global ids_map
    global new_id

    for entry in entries:
            image_path = entry[0]

            # Ugly but it works
            image_path = image_path.replace('\\', '/')
            image_path = image_path.replace('data/', '')
            image_path = image_path.replace('cuhk03', 'cuhk03/archive')
            

            image_name = os.path.basename(image_path)
            parsed_fname = image_name.split("_")
            identity = f"{parsed_fname[0]}_{parsed_fname[1]}"

            if identity in ids_map:
                identity = ids_map[identity]
            else:
                ids_map[identity] = f"{new_id}".zfill(4)
                identity = ids_map[identity]
                new_id += 1

            img_id = f"c{parsed_fname[2]}_{parsed_fname[3]}"

            if not os.path.exists(os.path.join(converted_path, f_type, identity)):
                os.makedirs(os.path.join(converted_path, f_type, identity))

            # Copy this image to the right path
            shutil.copy(image_path, os.path.join(converted_path, f_type, identity, f"{identity}_{img_id}"))

convert(gallery, "gallery")

In [45]:
# Move 20% of elements from gallery to query
percentage = 0.2

# Iterate over subfolders in the base directory
for folder_name in os.listdir(os.path.join(converted_path, "gallery")):
    folder_path = os.path.join(os.path.join(converted_path, "gallery"), folder_name)

    # Ensure it's a directory
    if not os.path.isdir(folder_path):
        continue

    print(f"Processing folder: {folder_name}")

    # List all files in the current folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

    # Shuffle files and select the top percentage to move
    num_to_move = int(len(files) * percentage)
    files_to_move = random.sample(files, num_to_move)

    for file_name in files_to_move:
        src_path = os.path.join(folder_path, file_name)

        dest_path = folder_path.replace("gallery", "query")

        # Create path for identity (0000, 0001, ...)
        if not os.path.exists(dest_path):
            os.makedirs(dest_path)
        
        dest_path = os.path.join(dest_path, file_name)

        # Move the file
        shutil.move(src_path, dest_path)
        print(f"Moved: {src_path} -> {dest_path}")

Processing folder: 0095
Moved: ./cuhk03/converted-split2/gallery/0095/0095_c2_08.png -> ./cuhk03/converted-split2/query/0095/0095_c2_08.png
Processing folder: 0061
Moved: ./cuhk03/converted-split2/gallery/0061/0061_c2_07.png -> ./cuhk03/converted-split2/query/0061/0061_c2_07.png
Processing folder: 0059
Moved: ./cuhk03/converted-split2/gallery/0059/0059_c2_07.png -> ./cuhk03/converted-split2/query/0059/0059_c2_07.png
Processing folder: 0066
Moved: ./cuhk03/converted-split2/gallery/0066/0066_c1_01.png -> ./cuhk03/converted-split2/query/0066/0066_c1_01.png
Moved: ./cuhk03/converted-split2/gallery/0066/0066_c2_10.png -> ./cuhk03/converted-split2/query/0066/0066_c2_10.png
Processing folder: 0092
Moved: ./cuhk03/converted-split2/gallery/0092/0092_c1_01.png -> ./cuhk03/converted-split2/query/0092/0092_c1_01.png
Moved: ./cuhk03/converted-split2/gallery/0092/0092_c2_07.png -> ./cuhk03/converted-split2/query/0092/0092_c2_07.png
Processing folder: 0050
Moved: ./cuhk03/converted-split2/gallery/005

# market-1501
We want to remove query images from the gallery (or avoid self-matching)

In [3]:
converted_path = "./Market-Pytorch/Market"

# Move 20% of elements from gallery to query
percentage = 0.2

# Iterate over subfolders in the base directory
for folder_name in os.listdir(os.path.join(converted_path, "gallery")):
    folder_path = os.path.join(os.path.join(converted_path, "gallery"), folder_name)

    # Ensure it's a directory and it's not called -1 or 0000
    if not os.path.isdir(folder_path) or folder_name in ["-1", "0000"]:
        continue

    print(f"Processing folder: {folder_name}")

    # List all files in the current folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

    # Shuffle files and select the top percentage to move
    num_to_move = int(len(files) * percentage)
    files_to_move = random.sample(files, num_to_move)

    for file_name in files_to_move:
        src_path = os.path.join(folder_path, file_name)

        dest_path = folder_path.replace("gallery", "query")

        # Create path for identity (0000, 0001, ...)
        if not os.path.exists(dest_path):
            os.makedirs(dest_path)
        
        dest_path = os.path.join(dest_path, file_name)

        # Move the file
        shutil.move(src_path, dest_path)
        print(f"Moved: {src_path} -> {dest_path}")

Processing folder: 1069
Moved: ./Market-Pytorch/Market/gallery/1069/1069_c2s2_139902_04.jpg -> ./Market-Pytorch/Market/query/1069/1069_c2s2_139902_04.jpg
Moved: ./Market-Pytorch/Market/gallery/1069/1069_c2s3_016207_03.jpg -> ./Market-Pytorch/Market/query/1069/1069_c2s3_016207_03.jpg
Moved: ./Market-Pytorch/Market/gallery/1069/1069_c5s3_022640_05.jpg -> ./Market-Pytorch/Market/query/1069/1069_c5s3_022640_05.jpg
Processing folder: 0719
Moved: ./Market-Pytorch/Market/gallery/0719/0719_c3s3_067669_01.jpg -> ./Market-Pytorch/Market/query/0719/0719_c3s3_067669_01.jpg
Moved: ./Market-Pytorch/Market/gallery/0719/0719_c2s3_061277_01.jpg -> ./Market-Pytorch/Market/query/0719/0719_c2s3_061277_01.jpg
Moved: ./Market-Pytorch/Market/gallery/0719/0719_c5s3_068312_02.jpg -> ./Market-Pytorch/Market/query/0719/0719_c5s3_068312_02.jpg
Processing folder: 1264
Moved: ./Market-Pytorch/Market/gallery/1264/1264_c6s3_055992_02.jpg -> ./Market-Pytorch/Market/query/1264/1264_c6s3_055992_02.jpg
Moved: ./Market-Py